# Phase 6 (post hoc): a physics prior for the generators and physiological fusion beats (phase6_physics_prior, P21)

PREREGISTRATION §10 v20, P21. SimGANs add McSharry et al.'s ECG simulator to a GAN's generator objective (the Euler loss).
P21 adds it to P12's cVAE and WGAN-GP (each generated patient-relative beat gets a real training patient's template before
it is scored), with its weight chosen on validation among 0.1, 1 and 10, and makes a third condition from physiology: P12's
cVAE banks with every synthetic F beat replaced by a fraction of a real training V deviation. Each condition goes through
P18's ten-run pipeline; Huang's stage and P16's veto are chosen on P14's patients by P20's rule; P19's patients and one DS2
read compare it with P18's cVAE or WGAN-GP.

| `PART` | accelerator | inputs | what it does |
|---|---|---|---|
| `check` | CPU | `ecgaug-repo`, `ecgaug-banks`, `ecgaug-v5-external` | fits the simulator to the training beats and the fusion model to the training F beats; the stop rules (`physics.json`) |
| `train` | GPU | as `check`, plus `ecgaug-p12-banks` | `FAMILY` = `cvae` or `wgan_gp`: seed 0 at each weight, the weight choice on validation, seeds 1-2, the ten cnn runs |
| `fusion` | GPU | as `train` | P12's cVAE banks with physiological F beats; the ten cnn runs |
| `score` | GPU | `ecgaug-repo`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs`; outputs of `phase6-rule-tuning-build` and the three P21 GPU sessions | the setting chosen on P14's patients; scores on P19's |
| `test` | CPU | `ecgaug-repo`, `ecgaug-v5-external`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs`; outputs of the three P21 GPU sessions | the one posthoc DS2 read (logged) |

`train` and `fusion` need `physics.json` from `check` in the repository; `score` needs the three GPU sessions' results;
`test` needs `choice.json` and P20's `test_summary.json`. Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
PART = "check"                        # "check" (CPU), "train" (GPU), "fusion" (GPU), "score" (GPU) or "test" (CPU, the DS2 read)
FAMILY = "cvae"                       # train only: "cvae" or "wgan_gp"
RUN_TESTS = False                     # the full test suite first; nothing else runs if it fails
INPUTS_DIR = "/kaggle/input"          # attached datasets: banks, checkpoints (runs/), external data (data/)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_physics_prior/physics_prior.yaml"]
extra = ["--part", PART, "--device", DEVICE]
if PART == "train":
    extra += ["--family", FAMILY]
if PART == "score":
    extra += ["--beats-dir", INPUTS_DIR]   # P20's built beats, from the attached phase6-rule-tuning-build output
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_physics_prior", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", INPUTS_DIR]
if PART != "score":
    cmd += ["--data-dir", INPUTS_DIR]   # not for score: the attached build output's repo/ folder holds a data/ folder too
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
if PART == "test":
    env["ALLOW_TEST_POSTHOC"] = "1"   # the one posthoc DS2 read of PREREGISTRATION v20; it is logged in docs/TEST_ACCESS_LOG.md
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
